# An Interpreter for a Simple Programming Language: Exercise Frame

This notebook is the starting point for the exercise at the end of the chapter *Implementing a Simple
Interpreter*.  It contains the interpreter developed in the notebook `01-Interpreter.ipynb`, extended by
*floating point numbers*, *strings*, and a number of *predefined functions*.  Your task is to extend this
interpreter as follows:

1. Add `for` loops to the interpreter.
2. Add the logical operators `&&` (*and*), `||` (*or*), and `!` (*not*).  The operator `!` should bind
   strongest and the operator `||` should bind weakest.
3. Add user-defined functions:
   - A function should only have access to its parameters and to those variables that are defined
     locally inside the function.
   - A function should always return a value using a `return` statement.  As a `return` statement can
     occur anywhere in a function, use an *exception* to communicate the return value and to transfer
     the control flow out of the function.

The places where you have to add code are marked with the comment `your code here`, both in this notebook and in
the grammar file `sl-frame.lark`.  The notebook runs as it is; only the example programs at the end need your
extensions.

This notebook uses the parser generator `Larkita`.
The parser generated by `Larkita` returns the abstract syntax tree of a program as a nested tuple.

In [1]:
from larkita import Larkita
import math

The external helper file `AST2Dot.ipynb` is used to represent the nested tuples that serve as our
abstract syntax trees as graphs.

In [2]:
%run AST2Dot.ipynb

In [3]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    import graphviz as gv
    type AST = str | int | float | tuple[str, *tuple["AST", ...]]
    def tuple2dot(t: AST) -> gv.Digraph: ...

## Specification of the Grammar

The grammar is stored in the file `sl-frame.lark`, which the cell below displays.  Compared to the grammar
`sl.lark` of `01-Interpreter.ipynb`, it has the following additions:
- Assignments are described by the syntactical variable `assign`, so that they can also be used inside
  the head of a `for` loop.
- The tokens `FLOAT` and `STRING` describe floating point numbers and string literals.

Add the grammar rules for `for` loops, the logical operators, function definitions, and `return`
statements to the file `sl-frame.lark`.  Remember that the grammar determines the abstract syntax tree:  The string literals of an alternative
form the *head* of the tuple that is returned for this alternative, while all other symbols become its children.
Since the directive `%ignore: "(", ")", "{", "}", ";"` removes the parentheses, the braces, and the semicolon from
all heads, an alternative of the form `"while" "(" bool_expr ")" "{" program "}"` yields the tuple
`('while', test, body)`, where `body` is the tuple `('program', stmnt, ...)`.  As for `if` and `while`, the bodies
of `for` loops and of function definitions have to be enclosed in braces.
If the head given by the literals is not what you want, give the alternative an *alias* `-> "name"`.  An alternative
that consists of a single symbol, like `assign ";"`, returns the tuple of this symbol.  For the logical operators,
introduce the syntactical variables `disjunction`, `conjunction`, and `negation` in order to encode the precedences.

In [4]:
from IPython.display import Code

Code(filename='sl-frame.lark', language='larkita')

program: stmnt*

stmnt: "if" "(" bool_expr ")" "{" program "}"
     | "while" "(" bool_expr ")" "{" program "}"
     // your code here: for loops
     // your code here: function definitions and return statements
     | assign ";"
     | expr ";"                           -> "expr"

assign: ID ":=" expr

// your code here: the logical operators ||, &&, and !
bool_expr: expr ("==" | "!=" | "<=" | ">=" | "<" | ">") expr

expr: expr ("+" | "-") product
    | product

product: product ("*" | "/" | "%") factor
       | factor

factor: "(" expr ")"
      | INT
      | FLOAT
      | STRING
      | ID
      | ID "(" [expr ("," expr)*] ")" -> "call"

// These literals are left out of the heads of all tuples.
%ignore: "(", ")", "{", "}", ";"

// Lexical definitions
%import common.CNAME -> ID, common.WS, common.CPP_COMMENT, common.C_COMMENT

FLOAT:  /[0-9](_?[0-9])*\.[0-9](_?[0-9])*([eE][+-]?[0-9]+)?/
INT:    /[0-9](_?[0-9])*/
STRING: /"(\\.|[^"\\])*"/

// Ignore whitespace and comments
%ignore WS, C_COMMENT, CPP_COMMENT

We generate the parser.  `Larkita` always generates a *canonical LR(1) parser*.  A description of the states of the
LR(1) automaton is written into the file `sl.out`.

In [ ]:
parser = Larkita.open('sl-frame.lark', out_file='sl.out')

## The Abstract Syntax Tree

The parser returns the abstract syntax tree as a nested tuple.  The leaves of this tree are tokens, which are
represented as pairs of the name of the token and its text:  `('INT', '42')`, `('FLOAT', '1.5')`,
`('ID', 'x')`, and `('STRING', '"abc"')`.  Note that the text of a string literal includes its double quotes.

In [ ]:
Number      = int | float
Value       = Number | str
NestedTuple = Value | tuple

The shapes of the tuples that are produced by the grammar above are the following:

| construct              | nested tuple                                       |
|:-----------------------|:---------------------------------------------------|
| program, body          | `('program', stmnt, ...)`                          |
| assignment             | `(':=', ('ID', name), expr)`                       |
| expression statement   | `('expr', expr)`                                   |
| conditional, loop      | `('if', test, body)`, `('while', test, body)`      |
| binary operator        | `(op, lhs, rhs)`, for example `('+', lhs, rhs)`    |
| function call          | `('call', ('ID', name), arg, ...)`                 |

For example, the following program yields the tuple shown below:

In [ ]:
parser.parse('x := 1.5 * y; print("x = ", x);')

The function `parse` takes a `file_name` as its sole argument.  The file is read and parsed into
an abstract syntax tree, which is then displayed as a graph.

In [ ]:
def parse(file_name: str):
    with open(file_name, 'r') as handle:
        program = handle.read()
    print(program)
    ast = parser.parse(program)
    print(ast)
    return tuple2dot(ast)

In [ ]:
parse('sum.sl')

## The Runtime System

A `return` statement can occur anywhere inside a function.  Therefore, it is implemented by raising an
exception of the class `ReturnValue`, which carries the value that is returned.  The call of the function
catches this exception.

In [ ]:
class ReturnValue(Exception):
    def __init__(self, value: Value):
        self.value = value

The dictionary `Functions` stores the user-defined functions.  It maps the name of a function to a pair
consisting of the list of its parameters and its body.

In [ ]:
Functions: dict[str, tuple[list[str], NestedTuple]] = {}

The function `execute_tuple` executes a list of statements.  The dictionary `Values` maps the names of
variables to their values.

In [ ]:
def execute_tuple(StatementList, Values: dict[str, Value]) -> None:
    for stmnt in StatementList:
        execute(stmnt, Values)

The function `execute` matches recursively against the shape of the nested tuple that represents a
statement.

In [ ]:
def execute(stmnt: NestedTuple, Values: dict[str, Value]) -> None:
    match stmnt:
        case ('program', *SL):
            execute_tuple(SL, Values)
        case (':=', ('ID', var), value):
            Values[var] = evaluate(value, Values)
        case ('expr', expr):
            evaluate(expr, Values)
        case ('if', test, stmnt):
            if evaluate_bool(test, Values):
                execute(stmnt, Values)
        case ('while', test, stmnt):
            while evaluate_bool(test, Values):
                execute(stmnt, Values)
        # your code here: for loops
        # your code here: function definitions
        # your code here: return statements (raise a ReturnValue exception)
        case _:
            assert False, f'{stmnt} unexpected'

The function `number` checks that a value is a number.  It is needed since our values may also be
strings, but the arithmetical operators, the comparison operators `<`, `<=`, `>`, `>=`, and the
mathematical functions are only defined for numbers.

In [ ]:
def number(value: Value) -> Number:
    assert not isinstance(value, str), f'{value!r} is not a number'
    return value

The function `evaluate_bool` evaluates the test of a conditional or of a loop and returns a Boolean
value.

In [ ]:
def evaluate_bool(expr: NestedTuple, Values: dict[str, Value]) -> bool:
    match expr:
        case ('==', lhs, rhs):
            return evaluate(lhs, Values) == evaluate(rhs, Values)
        case ('!=', lhs, rhs):
            return evaluate(lhs, Values) != evaluate(rhs, Values)
        case ('<=', lhs, rhs):
            return number(evaluate(lhs, Values)) <= number(evaluate(rhs, Values))
        case ('>=', lhs, rhs):
            return number(evaluate(lhs, Values)) >= number(evaluate(rhs, Values))
        case ('<', lhs, rhs):
            return number(evaluate(lhs, Values)) <  number(evaluate(rhs, Values))
        case ('>', lhs, rhs):
            return number(evaluate(lhs, Values)) >  number(evaluate(rhs, Values))
        # your code here: the logical operators &&, ||, and !
        case _:
            assert False, f'{expr} unexpected'

The function `evaluate` computes the value of an expression.  A call of a function is either a call of a
user-defined function or a call of a predefined function.
The leaves of the abstract syntax tree are tokens:  A token of type `INT` or `FLOAT` is converted into a number,
the text of a token of type `STRING` is stripped of its double quotes, and the value of a variable, i.e. of a token
of type `ID`, is looked up in `Values`.

In [ ]:
def evaluate(expr: NestedTuple, Values: dict[str, Value]) -> Value:
    match expr:
        case ('INT', digits):
            return int(digits)
        case ('FLOAT', digits):
            return float(digits)
        case ('STRING', text):
            return text[1:-1]
        case ('ID', name):
            return Values[name]
        case ('+', lhs, rhs):
            return number(evaluate(lhs, Values)) + number(evaluate(rhs, Values))
        case ('-', lhs, rhs):
            return number(evaluate(lhs, Values)) - number(evaluate(rhs, Values))
        case ('*', lhs, rhs):
            return number(evaluate(lhs, Values)) * number(evaluate(rhs, Values))
        case ('/', lhs, rhs):
            return number(evaluate(lhs, Values)) / number(evaluate(rhs, Values))
        case ('%', lhs, rhs):
            return number(evaluate(lhs, Values)) % number(evaluate(rhs, Values))
        case ('call', ('ID', f_name), *Args) if f_name in Functions:
            # your code here: call a user-defined function
            #  - evaluate the arguments,
            #  - execute the body with a new dictionary that contains only the parameters,
            #  - the body returns its value by raising a ReturnValue exception.
            raise NotImplementedError('calls of user-defined functions')
        case ('call', ('ID', f_name), *Args):
            return call_builtin(f_name, [evaluate(arg, Values) for arg in Args])
        case _:
            assert False, f'{expr} unexpected'

The function `call_builtin` implements the predefined functions of our language.  The function `print`
prints all of its arguments without separators between them.

In [ ]:
def call_builtin(f_name: str, Args: list[Value]) -> Value:
    match f_name:
        case 'read':
            s = input('Please enter a number: ')
            return float(s) if '.' in s or 'e' in s or 'E' in s else int(s)
        case 'print':
            print(*Args, sep='')
            return 0
        case 'sqrt':
            return math.sqrt(number(Args[0]))
        case 'exp':
            return math.exp(number(Args[0]))
        case 'ln':
            return math.log(number(Args[0]))
        case 'sin':
            return math.sin(number(Args[0]))
        case 'cos':
            return math.cos(number(Args[0]))
        case 'tan':
            return math.tan(number(Args[0]))
        case 'arctan':
            return math.atan(number(Args[0]))
        case _:
            assert False, f'function {f_name} is unknown'

## Putting Everything Together

The function `main` reads a program from a file, parses it, and executes it.  If `show_ast` is `True`, the
abstract syntax tree is displayed first.

In [ ]:
def main(file_name: str, show_ast: bool = False) -> None:
    with open(file_name, 'r') as handle:
        program = handle.read()
    ast = parser.parse(program)
    if show_ast:
        display(tuple2dot(ast))
    Functions.clear()
    execute(ast, {})

The program `sum.sl` only uses features that are already implemented.

In [ ]:
!cat sum.sl

In [ ]:
main('sum.sl')

## Testing Your Extensions

The following programs need your extensions.  The program `factorial.sl` uses user-defined functions and
a `for` loop.

In [ ]:
!cat factorial.sl

In [ ]:
main('factorial.sl')

In [ ]:
!cat e.sl

In [ ]:
main('e.sl')

In [ ]:
!cat pi.sl

In [ ]:
main('pi.sl')

In [ ]:
!cat solve.sl

In [ ]:
main('solve.sl')

In [ ]:
!cat sum-for.sl

In [ ]:
main('sum-for.sl')